In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import matplotlib.pyplot as plt
import torch
from PIL import Image

In [ ]:
# Select device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## Configuration

Same parameters as `run_sam3_text_prompt.py`

In [ ]:
# Paths - same as run_sam3_text_prompt.py defaults
EFFICIENTSAM3_ROOT = os.path.expanduser("~/workspace/code/Prompt/efficientsam3")

checkpoint_path = os.path.join(EFFICIENTSAM3_ROOT, "efficient_sam3_tinyvit_21m_mobileclip_s1.pth")
bpe_path = os.path.join(EFFICIENTSAM3_ROOT, "sam3/assets/bpe_simple_vocab_16e6.txt.gz")
image_path = os.path.join(EFFICIENTSAM3_ROOT, "sam3/assets/images/truck.jpg")

# Model configuration - matches run_sam3_text_prompt.py
backbone_type = "tinyvit"  # choices: tinyvit, efficientvit
model_name = "21m"  # e.g., 11m, 21m, b2
text_encoder_type = "MobileCLIP-S1"

# Text prompt
text_prompt = "truck"

print(f"Checkpoint: {checkpoint_path}")
print(f"BPE path: {bpe_path}")
print(f"Image: {image_path}")
print(f"Text prompt: '{text_prompt}'")

## Load Model (geti-prompt implementation)

In [ ]:
from getiprompt.models.foundation.efficientsam3.model_builder import build_efficientsam3_image_model
from getiprompt.models.foundation.sam3.sam3_image_processor import Sam3Processor

# Build model - same API as run_sam3_text_prompt.py
model = build_efficientsam3_image_model(
    checkpoint_path=checkpoint_path,
    bpe_path=bpe_path,
    backbone_type=f"{backbone_type}-{model_name}",  # geti-prompt uses combined format
    text_encoder_type=text_encoder_type,
    device=str(device),
)

print(f"Model loaded: {type(model).__name__}")

## Load Image

In [ ]:
image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.title(f"Input Image: {os.path.basename(image_path)}")
plt.axis("on")
plt.show()

print(f"Image size: {image.size}")

## Run Text Prompt Inference

This replicates the exact flow from `run_sam3_text_prompt.py`:
1. Create processor
2. Set image
3. Set text prompt
4. Get masks, scores, boxes

In [ ]:
# Create processor - same as run_sam3_text_prompt.py
processor = Sam3Processor(model)

# Set image
inference_state = processor.set_image(image)

# Set text prompt and run inference
inference_state = processor.set_text_prompt(text_prompt, inference_state)

# Get results - same as run_sam3_text_prompt.py
masks = inference_state["masks"]
scores = inference_state["scores"]
boxes = inference_state["boxes"]

# masks: Bool tensor with shape [N, 1, H, W]; scores: tensor with shape [N]
num_masks = int(masks.shape[0])
print(f"Number of masks: {num_masks}")
print(f"Scores: {scores.detach().cpu()}")

## Visualization

Similar visualization as `run_sam3_text_prompt.py`

In [ ]:
import numpy as np

# Colors from efficientsam3 visualization_utils
COLORS = [
    (0.000, 0.447, 0.741),  # blue
    (0.850, 0.325, 0.098),  # orange
    (0.929, 0.694, 0.125),  # yellow
    (0.494, 0.184, 0.556),  # purple
    (0.466, 0.674, 0.188),  # green
    (0.301, 0.745, 0.933),  # cyan
]


def plot_mask(mask, color, alpha=0.5):
    """Plot a mask with the given color."""
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * np.array([*color, alpha]).reshape(1, 1, 4)
    plt.gca().imshow(mask_image)


def plot_bbox(h, w, box, text, color):
    """Plot a bounding box with label."""
    x0, y0, x1, y1 = box
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=2)
    plt.gca().add_patch(rect)
    plt.text(
        x0,
        y0 - 5,
        text,
        color=color,
        fontsize=10,
        fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )

In [ ]:
# Visualization (matches run_sam3_text_prompt.py approach)
fig = plt.figure(figsize=(12, 8))
plt.imshow(image)
w, h = image.size

for i in range(num_masks):
    color = COLORS[i % len(COLORS)]

    # Plot mask
    mask_np = masks[i].squeeze(0).cpu().numpy().astype(np.float32)
    plot_mask(mask_np, color=color)

    # Plot bounding box
    prob = scores[i].item()
    box = boxes[i].cpu().numpy()
    plot_bbox(h, w, box, text=f"(id={i}, prob={prob:.2f})", color=color)

plt.title(f"Text Prompt: '{text_prompt}' - Found {num_masks} objects")
plt.axis("off")
plt.tight_layout()
plt.show()

if num_masks == 0:
    print("\nNo objects found. This could mean:")
    print("1. The confidence threshold is too high (try lowering it)")
    print("2. The checkpoint may not have grounding weights trained")
    print("3. The text prompt doesn't match objects in the image")

## Try with Lower Confidence Threshold

In [ ]:
# Try with lower confidence threshold
processor_low_conf = Sam3Processor(model, confidence_threshold=0.1)

inference_state_2 = processor_low_conf.set_image(image)
inference_state_2 = processor_low_conf.set_text_prompt(text_prompt, inference_state_2)

masks_2 = inference_state_2["masks"]
scores_2 = inference_state_2["scores"]
boxes_2 = inference_state_2["boxes"]

num_masks_2 = int(masks_2.shape[0])
print("With confidence_threshold=0.1:")
print(f"Number of masks: {num_masks_2}")
if num_masks_2 > 0:
    print(f"Scores: {scores_2.detach().cpu().tolist()}")

In [ ]:
# Visualize low confidence results
if num_masks_2 > 0:
    fig = plt.figure(figsize=(12, 8))
    plt.imshow(image)

    for i in range(num_masks_2):
        color = COLORS[i % len(COLORS)]
        mask_np = masks_2[i].squeeze(0).cpu().numpy().astype(np.float32)
        plot_mask(mask_np, color=color)

        prob = scores_2[i].item()
        box = boxes_2[i].cpu().numpy()
        plot_bbox(h, w, box, text=f"(id={i}, prob={prob:.2f})", color=color)

    plt.title(f"Text Prompt: '{text_prompt}' (threshold=0.1) - Found {num_masks_2} objects")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Still no objects found even with lower threshold.")

## Compare with Original efficientsam3 Implementation

Let's also test with the original efficientsam3 repo to compare results.

In [ ]:
import sys

# Add efficientsam3/sam3 to path to import original implementation
sam3_path = os.path.join(EFFICIENTSAM3_ROOT, "sam3")
if sam3_path not in sys.path:
    sys.path.insert(0, sam3_path)

# Import original implementation
from sam3.model.sam3_image_processor import Sam3Processor as Sam3ProcessorOriginal
from sam3.model_builder import build_efficientsam3_image_model as build_efficientsam3_original

print("Original efficientsam3 modules imported successfully")

In [ ]:
# Build original model
model_original = build_efficientsam3_original(
    checkpoint_path=checkpoint_path,
    backbone_type=backbone_type,  # Original uses separate backbone_type and model_name
    model_name=model_name,
    text_encoder_type=text_encoder_type,
)

print(f"Original model loaded: {type(model_original).__name__}")

In [ ]:
# Run inference with original implementation
processor_original = Sam3ProcessorOriginal(model_original)

inference_state_orig = processor_original.set_image(image)
inference_state_orig = processor_original.set_text_prompt(text_prompt, inference_state_orig)

masks_orig = inference_state_orig["masks"]
scores_orig = inference_state_orig["scores"]
boxes_orig = inference_state_orig["boxes"]

num_masks_orig = int(masks_orig.shape[0])
print("\n=== Original efficientsam3 Results ===")
print(f"Number of masks: {num_masks_orig}")
print(f"Scores: {scores_orig.detach().cpu()}")

In [ ]:
# Visualize original results
if num_masks_orig > 0:
    fig = plt.figure(figsize=(12, 8))
    plt.imshow(image)

    for i in range(num_masks_orig):
        color = COLORS[i % len(COLORS)]
        mask_np = masks_orig[i].squeeze(0).cpu().numpy().astype(np.float32)
        plot_mask(mask_np, color=color)

        prob = scores_orig[i].item()
        box = boxes_orig[i].cpu().numpy()
        plot_bbox(h, w, box, text=f"(id={i}, prob={prob:.2f})", color=color)

    plt.title(f"Original efficientsam3: '{text_prompt}' - Found {num_masks_orig} objects")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Original implementation also found no objects.")
    print("This confirms the checkpoint may not have text grounding weights.")

## Summary

This notebook tested the text encoder functionality comparing:
1. geti-prompt implementation
2. Original efficientsam3 implementation

Both should produce identical results if implemented correctly.